In [0]:
import pandas as pd

pdf = pd.read_parquet("../data/processed/data.parquet")
df = spark.createDataFrame(pdf)

# Análise exploratória

In [0]:
pdf.event.unique()

In [0]:
import plotly.express as px

steps = ['offer received', 'offer viewed', 'offer completed']
color_map = ['red', 'orange', 'green']
counts = [pdf[pdf['event'] == step].shape[0] for step in steps]

data = dict(
    number=counts,
    stage=steps)
fig = px.funnel(data, x='number', y='stage', color_discrete_sequence=color_map)
fig.show()

In [0]:
import plotly.express as px

# Define categories
received = pdf[pdf['event'] == 'offer received']
viewed = pdf[pdf['event'] == 'offer viewed']
completed = pdf[pdf['event'] == 'offer completed']

# Ensure matching on both offer_id and person for correct funnel logic
received_and_viewed = received.merge(viewed[['offer_id', 'account_id']], on=['offer_id', 'account_id'])
received_not_viewed = received.merge(viewed[['offer_id', 'account_id']], on=['offer_id', 'account_id'], how='left', indicator=True)
received_not_viewed = received_not_viewed[received_not_viewed['_merge'] == 'left_only'].drop(columns=['_merge'])

viewed_and_completed = viewed.merge(completed[['offer_id', 'account_id']], on=['offer_id', 'account_id'])
viewed_not_completed = viewed.merge(completed[['offer_id', 'account_id']], on=['offer_id', 'account_id'], how='left', indicator=True)
viewed_not_completed = viewed_not_completed[viewed_not_completed['_merge'] == 'left_only'].drop(columns=['_merge'])

# Prepare data
category_offer_type = pd.concat([
    received_and_viewed.assign(category='received_and_viewed'),
    received_not_viewed.assign(category='received_not_viewed'),
    viewed_and_completed.assign(category='viewed_and_completed'),
    viewed_not_completed.assign(category='viewed_not_completed')
])

offer_type_counts = category_offer_type.groupby(['category', 'offer_type']).size().reset_index(name='count')

fig = px.bar(
    offer_type_counts,
    x='category',
    y='count',
    color='offer_type',
    barmode='group',
    title='Offer Type per Category'
)
fig.show()

Ofertas informacionais não viram ofertas concluídas, mas estão com uma boa taxa de visualizações. Ofertas com descontos tem alta conversão, quando visualizadas. O gap maior está em aumentar as visualizações para este tipo de oferta. E as ofertas com pior conversão são as do tipo BOGO.

In [0]:
import plotly.express as px

# Plot distribution of min_value for each category
fig_min_value = px.box(
    category_offer_type,
    x='category',
    y='min_value',
    color='category',
    title='Distribution of Min Value by Category'
)
fig_min_value.show()

# Plot distribution of discount_value for each category
fig_min_value = px.box(
    category_offer_type,
    x='category',
    y='discount_value',
    color='category',
    title='Distribution of Discount Value by Category'
)
fig_min_value.show()


Valores mínimos muito altos, acima de 10, muitas vezes não são nem visualizados e valores muito baixos, menores que 5, são visualizados, porém não convertem. As ofertas completadas tem um valor que fica entre 5 e 10.

Um desconto maior, não necessariamente está ligado à maior conversão, como podemos observar na barra lilás que representa quem viu, mas não completou.

In [0]:
import plotly.express as px

# Plot distribution of duration for each category
fig_duration = px.box(
    category_offer_type,
    x='category',
    y='duration',
    color='category',
    title='Distribution of Duration by Category'
)
fig_duration.show()

Com relação à duração, é possível notar que as campanhas urgentes tem um grande apelo visual, mas não levam à conversão. 

In [0]:
import plotly.express as px

# Count channels per category
channels_per_category = category_offer_type.explode('channels').groupby(['category', 'channels']).size().reset_index(name='count')

fig = px.bar(
    channels_per_category,
    x='category',
    y='count',
    color='channels',
    barmode='group',
    title='Channels per Category'
)
fig.show()

Campanhas em rede social são as que dão mais engajamento, uma pequena parcela que recebe não visualiza essas ofertas. Mas quando se fala em conversão, elas ficam ligeiramente abaixo dos outros veículos, apesar da taxa de conversão ser similar em todos os veículos.

In [0]:
import plotly.express as px

fig_scatter = px.scatter(
    category_offer_type,
    x='min_value',
    y='discount_value',
    color='category',
    title='Min Value vs Discount Value by Category'
)
fig_scatter.show()

fig_scatter = px.scatter(
    category_offer_type,
    x='min_value',
    y='duration',
    color='category',
    title='Min Value vs Duration by Category'
)
fig_scatter.show()

Nenhuma correlação encontrada.

In [0]:
from sklearn.cluster import KMeans

# Feature selection
features = ['gender_num', 'age', 'credit_card_limit']
# , 'reward', 'min_value', 'duration', 'discount_value', 'offer_type_num']
X = pdf[features]

# Normalize data
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=4, random_state=42)
pdf['cluster'] = kmeans.fit_predict(X_scaled)

# Visualize clusters
fig_cluster = px.scatter(
    pdf,
    x='age',
    y='credit_card_limit',
    size='gender_num',
    color='cluster',
    title='Clusters de Clientes para Distribuição de Ofertas'
)
fig_cluster.show()

display(pdf.groupby('cluster')['event'].value_counts().unstack())

In [0]:
import plotly.express as px

# Select numeric columns
numeric_cols = pdf.select_dtypes(include='number').columns
corr = pdf[numeric_cols].corr()

fig_heatmap = px.imshow(
    corr,
    text_auto=True,
    color_continuous_scale='Viridis',
    title='Heatmap de Correlação das Variáveis Numéricas'
)
fig_heatmap.show()

# Modelo


Quero construir um modelo classificador em que todas as ofertas que foram completadas pertençam à classe 1, e as demais à classe 0. Para isso, é necessário remover todo o histórico da oferta e manter apenas o estágio final de cada jornada. Por exemplo: se uma oferta foi concluída, significa que ela foi vista e recebida anteriormente, ou seja, no dataset existem pelo menos três registros para essa oferta (um com event = received, outro com event = viewed e outro com event = completed). Precisamos remover esses registros intermediários para evitar data leakage e garantir que o modelo utilize apenas a informação do estágio final de cada oferta por cliente.

In [0]:
def get_final_stage_per_journey(df):
    """
    Retorna apenas o último evento de cada jornada (offer_id, account_id).
    Evita data leakage temporal mantendo apenas o estágio final alcançado: received, viewed ou completed.
    """
    # Sort by event
    event_order = {'offer received': 0, 'offer viewed': 1, 'offer completed': 2}
    df['event_rank'] = df['event'].map(event_order)
    
    # Get the highest rank for each offer
    final_stages = df.sort_values('event_rank').groupby(['offer_id', 'account_id']).tail(1).copy()
    
    return final_stages.drop('event_rank', axis=1)

modeling_df = get_final_stage_per_journey(pdf)

# Build target: 1 if completed, 0 otherwise
modeling_df['converted'] = (modeling_df['event'] == 'offer completed').astype(int)

In [0]:
print("="*80)
print("DATASET DE MODELAGEM")
print("="*80)
print(f"\nTotal de registros: {len(modeling_df):,}")
print(f"Total de registros originais: {len(pdf):,}")
print(f"Redução: {(1 - len(modeling_df)/len(pdf))*100:.1f}%")

print("\n" + "="*80)
print("DISTRIBUIÇÃO DO TARGET")
print("="*80)
print(modeling_df['converted'].value_counts())
print("\nProporção:")
print(modeling_df['converted'].value_counts(normalize=True).round(4))

print("\n" + "="*80)
print("DISTRIBUIÇÃO DOS ESTÁGIOS FINAIS")
print("="*80)
print(modeling_df['event'].value_counts())

print("\n" + "="*80)
print("VERIFICAÇÃO: Múltiplos registros por jornada")
print("="*80)
duplicate_check = modeling_df.groupby(['offer_id', 'account_id']).size()
duplicates = duplicate_check[duplicate_check > 1]
if len(duplicates) == 0:
    print("Cada jornada tem apenas 1 registro")
else:
    print(f"{len(duplicates)} jornadas com múltiplos registros:")
    print(duplicates.head())

Como o dataset está balanceado, podemos seguir para a escolha das features. Além de remover as colunas de identificadores, precisamos também remover aquelas que contém informação de futuro (event) para evitar data leakage. Optei por não usar o cluster já que não foi possível ter uma segmentação clara e ele não apresentou correlação com o evento. Além disso, vou criar duas features novas:
- Número de dias na plataforma, calculada a partir da data de registro
- Custo efetivo que mostra quantos reais efetivamente devem ser gastos para completar a oferta (valor mínimo - desconto)

In [0]:
modeling_df = modeling_df.drop(['event', 'event_num', 'account_id', 'time_since_test_start','offer_id', 'channels', 'offer_type', 'gender', 'cluster'], axis=1)

In [0]:
from datetime import date

modeling_df['days_on_plataform'] = modeling_df.registered_on.apply(lambda x: (date.today() - x).days)
modeling_df = modeling_df.drop(['registered_on'], axis=1)

modeling_df['effective_cost'] = modeling_df.min_value - modeling_df.discount_value

## Treinamento

In [0]:
from sklearn.model_selection import train_test_split

X = modeling_df.drop('converted', axis=1)
y = modeling_df['converted']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [0]:
# %pip install xgboost

In [0]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier
import time

models = {
    'RandomForest': (RandomForestClassifier(random_state=42), {
        'n_estimators': [50, 100],
        'max_depth': [5, 10]
    }),
    'LogisticRegression': (LogisticRegression(random_state=42, max_iter=1000, solver='saga'), {
        'C': [0.1, 1, 10],
        'penalty': ['l2']
    }),
    'XGBoost': (XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss'), {
        'n_estimators': [100, 200],
        'max_depth': [4, 6, 8],
        'learning_rate': [0.01, 0.1]
    })
}

results = {}
for name, (model, params) in models.items():
    print(f"\n{'='*80}")
    print(f"Treinando {name}...")
    print(f"{'='*80}")
    start_time = time.time()
    
    grid = GridSearchCV(
        model, 
        params, 
        cv=5, 
        scoring='accuracy', 
        n_jobs=-1,
        verbose=0
    )
    grid.fit(X_train, y_train)
    
    elapsed_time = time.time() - start_time
    results[name] = {
        'best_score': grid.best_score_,
        'best_params': grid.best_params_,
        'time': elapsed_time
    }
    print(f"✓ {name} concluído em {elapsed_time/60:.2f} minutos\n")

print("\n" + "="*80)
print("RESULTADOS FINAIS")
print("="*80)
for model_name, res in results.items():
    print(f"\n{model_name}:")
    print(f"  Best Score: {res['best_score']:.4f}")
    print(f"  Best Params: {res['best_params']}")
    print(f"  Tempo: {res['time']/60:.2f} minutos")

Esses resultados de previsão perfeita indicam overfitting. Vale investigar se ainda existe alguma feature com informação de futuro. Olhando novamente para o Heatmap, percebemos alta correlação entre reward e event (que é usado para montar o target). Vou investigar melhor essa feature e também amount, que são as duas informações que ainda vem do dataset de transações e podem conter dado futuro.

In [0]:
print("="*80)
print("FEATURES NO MODELO")
print("="*80)
print(f"Features: {list(X_train.columns)}")
print(f"\nTotal de features: {len(X_train.columns)}")

print("\n" + "="*80)
print("CORRELAÇÃO DAS FEATURES COM TARGET")
print("="*80)
correlations = X_train.join(y_train).corr()['converted'].sort_values(ascending=False)
print(correlations)

print("\n" + "="*80)
print("INSPEÇÃO: FEATURES SUSPEITAS (amount e reward)")
print("="*80)
# Verificar se amount e reward são sempre 0 para não-completados
print("\nDistribuição de 'amount' por classe:")
print(modeling_df.groupby('converted')['amount'].describe())

print("\nDistribuição de 'reward' por classe:")
print(modeling_df.groupby('converted')['reward'].describe())

print("\n" + "="*80)
print("ANÁLISE DETALHADA: amount e reward")
print("="*80)
print("\nValor médio de amount quando converted=0:", modeling_df[modeling_df['converted']==0]['amount'].mean())
print("Valor médio de amount quando converted=1:", modeling_df[modeling_df['converted']==1]['amount'].mean())
print("\nValor médio de reward quando converted=0:", modeling_df[modeling_df['converted']==0]['reward'].mean())
print("Valor médio de reward quando converted=1:", modeling_df[modeling_df['converted']==1]['reward'].mean())

print("\nValor total de amount quando converted=0:", modeling_df[modeling_df['converted']==0]['amount'].sum())
print("Valor total de amount quando converted=1:", modeling_df[modeling_df['converted']==1]['amount'].sum())
print("\nValor total de reward quando converted=0:", modeling_df[modeling_df['converted']==0]['reward'].sum())
print("Valor total de reward quando converted=1:", modeling_df[modeling_df['converted']==1]['reward'].sum())

A feature amount não está agregando em nada pois está totalmente zerada, então é melhor remover e a feature reward está zerada apenas para a classe 0, para registros da classe 1, ela tem valor e é aí que está o problema que leva a predição perfeita. Só existe valor em reward quando ocorreu uma conversão, isso é uma informação de futuro. Essa feature precisa ser removida também.

In [0]:
# Criar novo dataset sem leakage
modeling_df_clean = modeling_df.drop(['amount', 'reward'], axis=1)

X_clean = modeling_df_clean.drop('converted', axis=1)
y_clean = modeling_df_clean['converted']

X_train_clean, X_test_clean, y_train_clean, y_test_clean = train_test_split(
    X_clean, y_clean, test_size=0.2, random_state=42, stratify=y_clean
)

print(f"\nNovas dimensões:")
print(f"  X_train: {X_train_clean.shape}")
print(f"  X_test: {X_test_clean.shape}")
print(f"\nFeatures restantes: {list(X_train_clean.columns)}")
print(f"Total: {len(X_train_clean.columns)} features")

In [0]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier
import time

results_clean = {}
for name, (model, params) in models.items():
    print(f"\n{'='*80}")
    print(f"Treinando {name}...")
    print(f"{'='*80}")
    start_time = time.time()
    
    grid = GridSearchCV(
        model, 
        params, 
        cv=5, 
        scoring='accuracy', 
        n_jobs=-1,
        verbose=1
    )
    grid.fit(X_train_clean, y_train_clean)
    
    elapsed_time = time.time() - start_time
    
    train_score = grid.best_estimator_.score(X_train_clean, y_train_clean)
    test_score = grid.best_estimator_.score(X_test_clean, y_test_clean)
    gap = train_score - test_score
    
    results_clean[name] = {
        'best_score_cv': grid.best_score_,
        'best_params': grid.best_params_,
        'train_score': train_score,
        'test_score': test_score,
        'gap': gap,
        'time': elapsed_time
    }
    print(f" {name} concluído em {elapsed_time/60:.2f} minutos")
    print(f"  CV Score: {grid.best_score_:.4f}")
    print(f"  Train: {train_score:.4f} | Test: {test_score:.4f} | Gap: {gap:.4f}")

print("\n" + "="*80)
print("RESULTADOS FINAIS (SEM DATA LEAKAGE)")
print("="*80)
for model_name, res in results_clean.items():
    print(f"\n{model_name}:")
    print(f"  CV Score: {res['best_score_cv']:.4f}")
    print(f"  Train Accuracy: {res['train_score']:.4f}")
    print(f"  Test Accuracy: {res['test_score']:.4f}")
    print(f"  Gap: {res['gap']:.4f} {'(Overfitting!)' if res['gap'] > 0.05 else '(✓ OK)'}")
    print(f"  Best Params: {res['best_params']}")
    print(f"  Tempo: {res['time']/60:.2f} minutos")

Agora sim, temos um resultado mais coerente, sem informação de futuro e o melhor modelo encontrado foi o XGBoost.

In [0]:
import pandas as pd
import plotly.express as px
from xgboost import XGBClassifier

print("="*80)
print("ANÁLISE DE IMPORTÂNCIA DAS FEATURES - XGBOOST (MELHOR MODELO)")
print("="*80)

best_xgb = XGBClassifier(
    **results_clean['XGBoost']['best_params'],
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)
best_xgb.fit(X_train_clean, y_train_clean)

feature_importance = pd.DataFrame({
    'feature': X_train_clean.columns,
    'importance': best_xgb.feature_importances_
}).sort_values('importance', ascending=False)

fig = px.bar(
    feature_importance,
    x='importance',
    y='feature',
    orientation='h',
    title='Importância das Features - XGBoost',
    labels={'importance': 'Importância', 'feature': 'Feature'},
    color='importance',
    color_continuous_scale='Viridis'
)
fig.update_layout(height=500, showlegend=False)
fig.show()

print("\n" + "="*80)
print("TOP 5 FEATURES MAIS IMPORTANTES")
print("="*80)
for idx, row in feature_importance.head(5).iterrows():
    print(f"{row['feature']:.<30} {row['importance']:.4f} ({row['importance']/feature_importance['importance'].sum()*100:.1f}%)")

top_5_contrib = feature_importance.head(5)['importance'].sum() / feature_importance['importance'].sum() * 100
print(f"As 5 features mais importantes representam {top_5_contrib:.1f}% da importância total")

print("\n" + "="*80)
print("FEATURES MENOS IMPORTANTES")
print("="*80)
low_importance = feature_importance[feature_importance['importance'] < 0.05]
if len(low_importance) > 0:
    print(f"\n{len(low_importance)} features com importância < 5%:")
    print(low_importance['feature'].tolist())

In [0]:
from sklearn.metrics import roc_curve, auc, classification_report, confusion_matrix
import plotly.graph_objects as go
import plotly.express as px
import numpy as np

print("="*80)
print("AVALIAÇÃO DO MODELO - XGBOOST")
print("="*80)

y_pred_proba = best_xgb.predict_proba(X_test_clean)[:, 1]
y_pred = best_xgb.predict(X_test_clean)

# 1. ROC CURVE
fpr, tpr, thresholds = roc_curve(y_test_clean, y_pred_proba)
roc_auc = auc(fpr, tpr)

fig_roc = go.Figure()
fig_roc.add_trace(go.Scatter(
    x=fpr, y=tpr,
    mode='lines',
    name=f'XGBoost (AUC = {roc_auc:.3f})',
    line=dict(color='blue', width=2)
))
fig_roc.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1],
    mode='lines',
    name='Baseline (AUC = 0.500)',
    line=dict(color='red', width=2, dash='dash')
))
fig_roc.update_layout(
    title='Curva ROC - XGBoost',
    xaxis_title='Taxa de Falsos Positivos (FPR)',
    yaxis_title='Taxa de Verdadeiros Positivos (TPR)',
    width=700,
    height=600,
    showlegend=True
)
fig_roc.show()

print(f"\nAUC-ROC Score: {roc_auc:.4f}")
if roc_auc > 0.85:
    print("Excelente discriminação entre classes")
elif roc_auc > 0.75:
    print("Boa discriminação entre classes")
else:
    print("Discriminação moderada")

# 2. CONFUSION MATRIX
print("\n" + "="*80)
print("MATRIZ DE CONFUSÃO")
print("="*80)
cm = confusion_matrix(y_test_clean, y_pred)

fig_cm = px.imshow(
    cm,
    labels=dict(x="Predito", y="Real", color="Contagem"),
    x=['Não Converteu (0)', 'Converteu (1)'],
    y=['Não Converteu (0)', 'Converteu (1)'],
    text_auto=True,
    color_continuous_scale='Blues',
    title='Matriz de Confusão'
)
fig_cm.update_layout(width=600, height=500)
fig_cm.show()

tn, fp, fn, tp = cm.ravel()
print(f"\nVerdadeiros Negativos (TN): {tn:,}")
print(f"Falsos Positivos (FP): {fp:,}")
print(f"Falsos Negativos (FN): {fn:,}")
print(f"Verdadeiros Positivos (TP): {tp:,}")

# 3. CLASSIFICATION METRICS
print("\n" + "="*80)
print("MÉTRICAS DE CLASSIFICAÇÃO")
print("="*80)
print(classification_report(y_test_clean, y_pred, target_names=['Não Converteu', 'Converteu']))

# 4. BUSINESS METRICS
print("\n" + "="*80)
print("MÉTRICAS DE NEGÓCIO")
print("="*80)
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
accuracy = (tp + tn) / (tp + tn + fp + fn)

print(f"Precision (Converteu): {precision:.4f}")
print(f"  → De todas as predições de conversão, {precision*100:.1f}% estavam corretas")
print(f"\nRecall (Converteu): {recall:.4f}")
print(f"  → De todas as conversões reais, o modelo identificou {recall*100:.1f}%")
print(f"\nSpecificity (Não Converteu): {specificity:.4f}")
print(f"  → De todos os não-conversões reais, o modelo identificou {specificity*100:.1f}%")
print(f"\nF1-Score: {f1:.4f}")
if f1 > 0.5:
    print(f"  → Bom equilibrio entre pecisão e recall")
else:
    print(f"  → Ruim equilibrio entre pecisão e recall")
print(f"\nAccuracy: {accuracy:.4f}")
print(f"  → De todos as predições, o modelo acertou {accuracy*100:.1f}%")

# 5. PROBABILITIES DISTRIBUTION
print("\n" + "="*80)
print("DISTRIBUIÇÃO DAS PROBABILIDADES PREDITAS")
print("="*80)
proba_df = pd.DataFrame({
    'probabilidade': y_pred_proba,
    'classe_real': ['Converteu' if y == 1 else 'Não Converteu' for y in y_test_clean]
})

fig_dist = px.histogram(
    proba_df,
    x='probabilidade',
    color='classe_real',
    nbins=50,
    title='Distribuição das Probabilidades Preditas por Classe Real',
    labels={'probabilidade': 'Probabilidade de Conversão', 'count': 'Frequência'},
    barmode='overlay',
    opacity=0.7
)
fig_dist.update_layout(width=800, height=500)
fig_dist.show()

print("\nProbabilidade média predita para classe 0:", proba_df[proba_df['classe_real']=='Não Converteu']['probabilidade'].mean())
print("Probabilidade média predita para classe 1:", proba_df[proba_df['classe_real']=='Converteu']['probabilidade'].mean())

Agora que temos um modelo, podemos seguir para um próximo passo que seria definir uma arquitetura para deploy. Defini uma arquitetura assíncrona para evitar problemas de latência e proponho um mecanismo de disparo de notificação via Slack, já que o iFood não usa email.
![Arquitetura](../Architecture.png)